# PRACTICAL 3: Data Cleaning and Preprocessing

## Objective
Load the structured dataset from `structured_logs.csv` and perform cleaning and preprocessing steps to make the logs suitable for machine learning features. Specifically, we will:
1. **Safely convert datetime column**: Safely parse `timestamp` into a pandas datetime object, handling parsing errors resiliently.
2. **Handle missing values**: Standardize missing representation formats (e.g. `NaN`, `null`, `None`, `-`) into `pd.NA` and fill appropriately.
3. **Text normalization**: Clean up whitespace, lowercase, and normalize values in categorical features.
4. **URL path normalization**: Standardize requesting resources by removing trailing slashes and common extensions (like `.html`).

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

# Set aesthetics styling
plt.style.use('seaborn-v0_8')
%matplotlib inline

### 1. Load Structured Dataset

In [ ]:
input_file = "structured_logs.csv"
print(f"Loading dataset from '{input_file}'...")
df = pd.read_csv(input_file, low_memory=False)
initial_count = len(df)
print(f"Initial Total Rows Loaded: {initial_count:,}")

### 2. Safely Convert Timestamp Column

In [ ]:
print("Converting 'timestamp' column to Datetime format...")
if 'timestamp' in df.columns:
    df['timestamp'] = df['timestamp'].astype(str).str.replace('"', '').str.strip()
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    
    invalid_count = df['timestamp'].isna().sum()
    if invalid_count > 0:
        print(f"  ⚠️ Warning: {invalid_count:,} timestamps could not be parsed. Filling with default.")
        df['timestamp'] = df['timestamp'].fillna(pd.Timestamp('2023-01-08 00:00:00'))

### 3. Handle Missing Values & Normalize String Fields

In [ ]:
print("Cleaning missing values and normalizing text...")
df.replace(['null', 'NaN', 'nan', 'None', '-'], pd.NA, inplace=True)

text_columns = ['resource_requested', 'user_agent', 'referrer', 'request_type']
for col in text_columns:
    if col in df.columns:
        df[col] = df[col].fillna('unknown').astype(str).str.lower().str.strip()
        
if 'ip' in df.columns:
    df['ip'] = df['ip'].fillna('0.0.0.0').astype(str).str.strip()

### 4. Normalize URL Resources

In [ ]:
print("Normalizing URL paths...")
if 'resource_requested' in df.columns:
    df['resource_requested'] = df['resource_requested'].str.rstrip('/')
    df['resource_requested'] = df['resource_requested'].str.replace(r'\.html$', '', regex=True)
    df.loc[df['resource_requested'] == '', 'resource_requested'] = '/'

### 5. Export Cleaned Dataset & Save Status Visualization

In [ ]:
output_file = "cleaned_logs.csv"
df.to_csv(output_file, index=False)
print(f"Saved cleaned dataset to '{output_file}'!")

# Status code visualization
os.makedirs("../plots", exist_ok=True)
plt.figure(figsize=(10, 6))
if 'status_code' in df.columns:
    status_counts = df['status_code'].value_counts()
    sns.barplot(x=status_counts.index, y=status_counts.values, palette='viridis', hue=status_counts.index, legend=False)
    plt.title("Distribution of HTTP Status Codes in Cleaned Logs", fontsize=14, fontweight='bold', pad=15)
    plt.xlabel("HTTP Status Code", fontsize=12)
    plt.ylabel("Request Count", fontsize=12)
    plt.yscale('log')
    for i, val in enumerate(status_counts.values):
        plt.text(i, val * 1.1, f"{val:,}", ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig("../plots/plot_p3_status_codes.png", dpi=300)
plt.show()

### 6. Review Dataset Summary Info

In [ ]:
print("--- Cleaned DataFrame Info ---")
df.info()

print("\n--- First 5 Rows of Cleaned Dataset ---")
df.head()